In [ ]:
# Data processing libraries
import json

# MLflow Library Imports and OS Directory Handling
import os

# Pipeline serialization library (used for exports & imports)
import joblib

# Data visualization
import matplotlib.pyplot as plt
import mlflow.sklearn
import numpy as np
import pandas as pd
import scipy as sp
import seaborn as sns
from mlflow import (
    get_tracking_uri,
    log_metric,
    log_params,
    set_experiment,
    set_tracking_uri,
    start_run,
)
from mlflow.models.signature import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Metric to track and log for MLflow
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Manual Grid Search
from sklearn.model_selection import ParameterGrid, train_test_split

# Sklearn libraries for preprocessing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder

# XGBoost Classifier Algorithm
from xgboost import XGBClassifier

import dataprocessing as dc


In [ ]:
datafilename = "email_data.csv"

df = pd.read_csv(f"data/{datafilename}")
newdata = dc.Data_class(data = df, )
df = newdata.data
df["Category"] = np.where(df['Category'] == 'spam', 0, 1)

In [ ]:
y = df["Category"]
X = df.drop(
    ["Category", "Message", "word_count"],
    axis = 1
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    stratify= y,
    random_state = 42
)


In [ ]:
numeric_cols = X.select_dtypes(
    include = ["number"]
).columns
numeric_pipe = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="mean")),
        ("encoder",MinMaxScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, numeric_cols)
    ],
    remainder='drop'  # or 'passthrough' if you want to keep unlisted columns
)

In [ ]:
%%capture

# Set experiment name for this module
set_experiment("XGBoost")

# Specify hyperparameters
hyperparam_grid = {
    "n_estimators": [50, 200],  # Total number of trees in ensemble
    "max_depth": [3, 10],     # Max tree depth
    "learning_rate": [0.05, 0.1, 0.5]   # Different learning rates
}

# Iterate through all hyperparameter value configurations
for hyperparams in ParameterGrid(hyperparam_grid):
    with start_run(run_name = "XGBoost Grid Search", nested = True):

        # Define XGBoost model in pipeline
        xgboost_model = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", XGBClassifier(
                eval_metric="logloss",
                **hyperparams
            ))
        ])

        # Fit model to training data, evaluate on holdout (testing) data
        xgboost_model.fit(X_train, y_train)

        y_pred = xgboost_model.predict(X_test)
        y_proba = xgboost_model.predict_proba(X_test)[:, 1]

        acc       = accuracy_score(y_test, y_pred)
        pr_auc    = average_precision_score(y_test, y_proba)
        roc_auc   = roc_auc_score(y_test, y_proba)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall    = recall_score(y_test, y_pred, zero_division=0)
        f1        = f1_score(y_test, y_pred, zero_division=0)

        # Log parameters & metrics
        log_params(hyperparams)
        log_metric("accuracy", acc)
        log_metric("Average Precision", pr_auc)
        log_metric("roc_auc", roc_auc)
        log_metric("precision", precision)
        log_metric("recall", recall)
        log_metric("f1", f1)

        # Optional but best practice: define input example and signature
        input_example = pd.DataFrame(X_test[:1], columns=X_test.columns)
        signature = infer_signature(X_test, y_proba)   # scores rather than labels

        # Log candidate model in MLflow experiment
        mlflow.sklearn.log_model(
            xgboost_model, name="candidate_model",
            input_example=input_example,
            signature=signature,
            serialization_format="cloudpickle"
        )

In [ ]:
!mlflow ui